# CatVTON × MAEVEN — chạy thử virtual try-on trên Kaggle

Bước 1 trong lộ trình học (`research/README.md`). Mục tiêu: hiểu trọn quy trình **tách người → mask vùng áo → diffusion**, và xem model xử lý vải lanh, bông dệt chéo của MAEVEN ra sao.

> **License:** code và weights của CatVTON là **CC BY-NC-SA 4.0**. Chỉ dùng để học và thử nghiệm. **Không** đưa ảnh tạo ra lên site bán hàng.

### Thiết lập notebook trên Kaggle
1. *Settings → Accelerator*: **GPU T4 x2** hoặc **GPU P100**.
2. *Settings → Internet*: **On** (cần tài khoản đã xác minh số điện thoại). Notebook tải weights từ Hugging Face.
3. *Add Input*: dataset ảnh của bạn (cấu trúc ở mục 4). Chưa có thì cứ chạy, mục 3 dùng ảnh mẫu của repo.


## 1. Cài đặt
Phiên bản `diffusers` / `transformers` được ghim theo đúng bản repo đã test, để tránh lỗi do API thay đổi.

In [ ]:
!git clone -q --depth 1 https://github.com/Zheng-Chong/CatVTON.git /kaggle/working/CatVTON
!pip install -q "diffusers==0.36.0" "transformers==4.46.3" "accelerate>=0.31" "peft>=0.17" \
    "huggingface_hub>=0.34,<2.0" fvcore omegaconf pycocotools cloudpickle av

In [ ]:
import os, sys, time, json
from pathlib import Path

REPO = Path("/kaggle/working/CatVTON")
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import torch
from PIL import Image
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download

from model.cloth_masker import AutoMasker, vis_mask
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

assert torch.cuda.is_available(), "Bật GPU trong Settings → Accelerator"
# bf16 chỉ có phần cứng từ Ampere (sm80) trở lên. Không dùng is_bf16_supported(): bản PyTorch mới
# trả True cả trên T4 vì tính bf16 giả lập, khiến model chạy rất chậm và mất kernel attention tiết kiệm bộ nhớ.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(torch.cuda.get_device_name(0), DTYPE)

## 2. Tải model

Có ba phần:
- **Stable Diffusion inpainting**: model nền.
- **Attention weights của CatVTON** (`mix-48k-1024`): phần duy nhất được train, khoảng 49,57M tham số.
- **DensePose + SCHP**: hai model phụ để biết đâu là thân người, đâu là áo, từ đó tạo mask.

CatVTON **không dùng text encoder**. Ảnh người và ảnh áo được ghép cạnh nhau rồi đưa qua cùng một UNet. Đó là ý "concatenation is all you need" trong tên paper.

In [ ]:
ckpt = snapshot_download(repo_id="zhengchong/CatVTON")

pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",  # bản sao, repo gốc của runwayml đã bị xoá
    attn_ckpt=ckpt,
    attn_ckpt_version="mix",
    weight_dtype=DTYPE,
    use_tf32=True,
    device="cuda",
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(ckpt, "DensePose"),
    schp_ckpt=os.path.join(ckpt, "SCHP"),
    device="cuda",
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
print(f"VRAM sau khi nạp: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
import gc
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

# Ảnh người + áo ghép lại thành 24.576 điểm. Nếu PyTorch rơi về cách tính "math", nó tạo cả ma trận
# attention ở fp32 (~36 GB) và tràn bộ nhớ T4. Ép dùng kernel memory-efficient; không được thì chia khúc.
if not hasattr(F, "_sdpa_original"):
    F._sdpa_original = F.scaled_dot_product_attention
_sdpa = F._sdpa_original
_reported = set()

def sdpa_low_mem(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):
    try:
        with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
            out = _sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)
        if "efficient" not in _reported:
            _reported.add("efficient"); print("attention: memory-efficient kernel")
        return out
    except RuntimeError:
        pass
    if "chunked" not in _reported:
        _reported.add("chunked"); print("attention: chia khúc (chậm hơn nhưng không tràn bộ nhớ)")
    out = torch.empty_like(query)
    step = 1024
    for i in range(0, query.shape[-2], step):
        m = attn_mask[..., i:i + step, :] if attn_mask is not None else None
        out[..., i:i + step, :] = _sdpa(query[..., i:i + step, :], key, value, attn_mask=m, dropout_p=dropout_p, **kwargs)
    return out

F.scaled_dot_product_attention = sdpa_low_mem
gc.collect(); torch.cuda.empty_cache()

W, H = 768, 1024
OUT = Path("/kaggle/working/out")
OUT.mkdir(exist_ok=True)

def try_on(person_path, garment_path, cloth_type="upper", steps=50, guidance=2.5, seed=42):
    """Giống submit_function trong app.py của repo, bỏ phần Gradio.

    cloth_type: 'upper' | 'lower' | 'overall' | 'inner' | 'outer'
    Trả về (ảnh kết quả, ảnh xem mask, số giây).
    """
    person = resize_and_crop(Image.open(person_path).convert("RGB"), (W, H))
    garment = resize_and_padding(Image.open(garment_path).convert("RGB"), (W, H))

    mask = automasker(person, cloth_type)["mask"]
    mask = mask_processor.blur(mask, blur_factor=9)

    generator = torch.Generator(device="cuda").manual_seed(seed)
    torch.cuda.synchronize(); t0 = time.time()
    result = pipeline(
        image=person,
        condition_image=garment,
        mask=mask,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
    )[0]
    torch.cuda.synchronize()
    return result, vis_mask(person, mask), time.time() - t0

def strip(images, height=512):
    """Ghép các ảnh thành một hàng ngang để so sánh."""
    images = [im.resize((int(im.width * height / im.height), height)) for im in images]
    out = Image.new("RGB", (sum(im.width for im in images), height), "white")
    x = 0
    for im in images:
        out.paste(im, (x, 0)); x += im.width
    return out

## 3. Chạy thử bằng ảnh mẫu của repo
Cho chắc model chạy đúng trước khi dùng ảnh của bạn. Nếu ảnh ra **đen hoàn toàn** thì safety checker đã chặn nhầm. Đổi `seed` rồi chạy lại.

In [ ]:
person = REPO / "resource/demo/example/person/men/model_5.png"
garment = next((REPO / "resource/demo/example/condition/upper").glob("*.jpg"))

result, masked, secs = try_on(person, garment, "upper")
print(f"{secs:.1f}s · VRAM đỉnh {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
strip([Image.open(person), masked, Image.open(garment), result])

## 4. Ảnh của MAEVEN

Tạo một Kaggle Dataset (ví dụ `maeven-tryon`) theo cấu trúc:

```
maeven-tryon/
  garments/   mv-01.jpg  mv-02.jpg  mv-03.jpg  mv-04.jpg   ← ảnh sản phẩm THẬT, nền trơn, trải phẳng hoặc treo
  persons/    slim.jpg   regular.jpg   broad.jpg ...       ← người mẫu đứng thẳng, chính diện, thấy rõ thân
```

`GARMENTS` cho model biết mỗi món thuộc vùng nào trên cơ thể. Áo khoác (`mv-03`) thử cả `outer` lẫn `upper` xem cái nào đẹp hơn.

In [ ]:
import zipfile

GARMENTS = {
    "mv-01": "upper",  # Áo sơ mi lanh mộc
    "mv-02": "lower",  # Quần ống suông
    "mv-03": "outer",  # Áo khoác không cổ
    "mv-04": "upper",  # Áo thun cổ tròn
}
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

def find_dataset(root=Path("/kaggle/input")):
    # Tìm thư mục chứa garments/ ở mọi độ sâu. Kaggle có thể gắn dataset sâu hơn một tầng,
    # hoặc giữ nguyên file zip; khi đó tự giải nén vào /kaggle/working/data.
    hits = sorted(p.parent for p in root.rglob("garments") if p.is_dir())
    if hits:
        return hits[0]
    for z in root.rglob("*.zip"):
        out = Path("/kaggle/working/data") / z.stem
        with zipfile.ZipFile(z) as f:
            f.extractall(out)
        hits = sorted(p.parent for p in out.rglob("garments") if p.is_dir())
        if hits:
            return hits[0]
    return None

def show_input_tree(root=Path("/kaggle/input"), limit=40):
    print("Không thấy thư mục garments/. Những gì đang có trong /kaggle/input:")
    for p in sorted(root.rglob("*"))[:limit]:
        print("  ", p)

DATA = find_dataset()
if DATA is None:
    show_input_tree()
    print("Thêm dataset qua 'Add Input' rồi chạy lại cell này.")
elif not (DATA / "persons").is_dir():
    print(f"{DATA} thiếu thư mục persons/")
    DATA = None
else:
    garments = {p.stem: p for p in sorted((DATA / "garments").iterdir()) if p.suffix.lower() in IMG_EXT}
    persons = [p for p in sorted((DATA / "persons").iterdir()) if p.suffix.lower() in IMG_EXT]
    missing = set(garments) - set(GARMENTS)
    print(f"{len(garments)} sản phẩm, {len(persons)} người mẫu từ {DATA}")
    if missing:
        print("Chưa khai báo vùng cho:", sorted(missing))

In [ ]:
log = []
if DATA is not None:
    for sku, gpath in garments.items():
        if sku not in GARMENTS:
            continue
        row = []
        for ppath in persons:
            result, _, secs = try_on(ppath, gpath, GARMENTS[sku])
            name = f"{sku}__{ppath.stem}.jpg"
            result.save(OUT / name, quality=92)
            log.append({"sku": sku, "person": ppath.stem, "type": GARMENTS[sku], "seconds": round(secs, 1), "file": name})
            row.append(result)
        display(strip([Image.open(gpath)] + row, height=384))

    (OUT / "log.json").write_text(json.dumps(log, ensure_ascii=False, indent=2))
    print(f"{len(log)} ảnh, trung bình {np.mean([r['seconds'] for r in log]):.1f}s/ảnh → {OUT}")

## 5. Thí nghiệm: steps và guidance

Hai tham số đáng hiểu nhất:
- **`num_inference_steps`**: số bước khử nhiễu. Nhiều bước thì chi tiết hơn nhưng chậm hơn, thời gian tăng gần như tuyến tính.
- **`guidance_scale`**: mức bám theo ảnh áo. Thấp thì mềm và tự nhiên nhưng dễ lệch họa tiết. Cao thì bám sát hơn nhưng dễ bị gắt và quá bão hoà.

In [ ]:
if DATA is not None and garments and persons:
    sku = next(iter(garments))
    p, g = persons[0], garments[sku]
    for steps in (20, 30, 50):
        row, times = [], []
        for guidance in (1.5, 2.5, 3.5):
            r, _, s = try_on(p, g, GARMENTS.get(sku, "upper"), steps=steps, guidance=guidance)
            row.append(r); times.append(s)
        print(f"steps={steps}  guidance 1.5 / 2.5 / 3.5  ·  {' / '.join(f'{t:.1f}s' for t in times)}")
        display(strip(row, height=384))

## 6. Chấm kết quả

Ghi vào `research/README.md` (mục *Nhật ký thí nghiệm*) cho từng sản phẩm:

| Tiêu chí | Câu hỏi |
|---|---|
| Màu | Có lệch so với áo thật không? Khách đã phàn nàn "màu ngoài đời nhạt hơn web". |
| Chất vải | Nếp nhăn tự nhiên của lanh còn không, hay bị làm phẳng như vải tổng hợp? |
| Phom | Ống suông còn suông không? Cổ bo của áo thun có bị biến dạng? |
| Danh tính | Khuôn mặt, tay và nền ảnh có giữ nguyên không? |
| Thời gian | Bao nhiêu giây mỗi ảnh trên T4? |

Muốn tải ảnh về: *Output → `/kaggle/working/out`*.